# Get the top 100 games from the steam charts

SteamSpy API Notes  
  *IMPORTANT:* some things have changed, please, read this document through!

  The data is refreshed once a day, there is no reason to request the same information more than once every 24 hours.

  Allowed poll rate - 1 request per second for most requests, 1 request per 60 seconds for the *all* requests.

In [1]:
import requests
import pandas as pd


# Get top 100 games by player count from SteamSpy
response = requests.get('https://steamspy.com/api.php?request=top100in2weeks')
top_games = pd.DataFrame(response.json()).T #shortand for JSON transpose rows to cols

In [2]:
top_games.head()

,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
730,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",31031,837,5370,326,0,0,0,1013936
1172470,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",9256,642,743,226,0,0,0,124262
578080,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",23259,949,6088,399,0,0,0,314682
1623730,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",3684,900,2213,406,2999,2999,0,18028
440,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",9863,731,354,120,0,0,0,43819


We have an interesting list of the top 100 games from the last 2 weeks!  
Most importantly it contains the appid we can use to make more data requests.

In [3]:
top_games = top_games.head(100) #get first 100 rows
top_games['appid'] = top_games['appid'].astype(int) #convert appid to integer
top_games[['appid', 'name']].head() #get first 5 rows of appid and name

,appid,name
730,730,Counter-Strike: Global Offensive
1172470,1172470,Apex Legends
578080,578080,PUBG: BATTLEGROUNDS
1623730,1623730,Palworld
440,440,Team Fortress 2


In [4]:
# Steamspy can get per-appid data as well
id_response = requests.get('https://steamspy.com/api.php?request=appdetails&appid=730')

In [5]:
# Example Data returned
id_response.json()

{'appid': 730,
 'name': 'Counter-Strike: Global Offensive',
 'developer': 'Valve',
 'publisher': 'Valve',
 'score_rank': '',
 'positive': 7642084,
 'negative': 1173003,
 'userscore': 0,
 'owners': '100,000,000 .. 200,000,000',
 'average_forever': 31031,
 'average_2weeks': 837,
 'median_forever': 5370,
 'median_2weeks': 326,
 'price': '0',
 'initialprice': '0',
 'discount': '0',
 'ccu': 1013936,
 'languages': 'English, Czech, Danish, Dutch, Finnish, French, German, Hungarian, Italian, Japanese, Korean, Norwegian, Polish, Portuguese - Portugal, Portuguese - Brazil, Romanian, Russian, Simplified Chinese, Spanish - Spain, Swedish, Thai, Traditional Chinese, Turkish, Bulgarian, Ukrainian, Greek, Spanish - Latin America, Vietnamese, Indonesian',
 'genre': 'Action, Free To Play',
 'tags': {'FPS': 91172,
  'Shooter': 65634,
  'Multiplayer': 62536,
  'Competitive': 53536,
  'Action': 47634,
  'Team-Based': 46549,
  'e-sports': 43682,
  'Tactical': 41468,
  'First-Person': 39540,
  'PvP': 34587,

For the future - the sales and tags data could be analyzed to create value.

# Grabbing reviews from Steam

In [6]:
!pip install -Uqq steam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.1/644.1 kB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [7]:
import os
from steam.webapi import WebAPI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
STEAM_API_KEY = user_secrets.get_secret("STEAM_API_KEY")

api = WebAPI(key=STEAM_API_KEY)

# Example: Get reviews for a single app
def get_reviews(appid, num_reviews=100, cursor='*'):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1,
        'num_per_page': num_reviews,
        'cursor': cursor,
        'filter': 'recent',
        'language': 'english'
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if 'reviews' in data:
            return { 'cursor': data['cursor'], 'reviews': data['reviews'] }
    return []

In [8]:
# Preview of CS review JSON data
cs_reviews = get_reviews(730, num_reviews=1)
cs_reviews['reviews'][0]

{'recommendationid': '200623993',
 'author': {'steamid': '76561199807551165',
  'num_games_owned': 0,
  'num_reviews': 1,
  'playtime_forever': 16108,
  'playtime_last_two_weeks': 4203,
  'playtime_at_review': 16073,
  'last_played': 1753400290},
 'language': 'english',
 'review': 'top game no cheaters',
 'timestamp_created': 1753398278,
 'timestamp_updated': 1753398278,
 'voted_up': True,
 'votes_up': 0,
 'votes_funny': 0,
 'weighted_vote_score': 0.5,
 'comment_count': 0,
 'steam_purchase': True,
 'received_for_free': False,
 'written_during_early_access': False,
 'primarily_steam_deck': False}

In [9]:
from tqdm import tqdm

all_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    appid = row['appid']
    name = row['name']
    response = get_reviews(appid, num_reviews=10)
    for review in response['reviews']:
        all_reviews.append({
            'appid': appid,
            'name': name,
            'review': review['review'],
            'timestamp_created': review['timestamp_created'],
            'voted_up': review['voted_up'],
            'votes_up': review['votes_up'],
            'votes_funny': review['votes_funny'],
            'weighted_vote_score': review['weighted_vote_score'],
        })

reviews_df = pd.DataFrame(all_reviews)
reviews_df.head()

100%|██████████| 100/100 [00:21<00:00,  4.67it/s]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,top game no cheaters,1753398278,True,0,0,0.5
1,730,Counter-Strike: Global Offensive,"Good game, there are nuances, yes. As in any o...",1753397848,True,1,0,0.523809552192687988
2,730,Counter-Strike: Global Offensive,https://www.youtube.com/watch?v=Fqv_qIuyeuU,1753397629,True,0,0,0.5
3,730,Counter-Strike: Global Offensive,Cheat feast,1753397198,False,0,0,0.5
4,730,Counter-Strike: Global Offensive,ytr,1753397066,True,0,0,0.5


In [10]:
reviews_df.to_csv('reviews.csv', index=False)

# Start Here for analysis

Provided the dataframe is saved from above - this is an entry point to 1000 random reviews

In [11]:
import pandas as pd

df = pd.read_csv('reviews.csv')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   appid                1000 non-null   int64  
 1   name                 1000 non-null   object 
 2   review               996 non-null    object 
 3   timestamp_created    1000 non-null   int64  
 4   voted_up             1000 non-null   bool   
 5   votes_up             1000 non-null   int64  
 6   votes_funny          1000 non-null   int64  
 7   weighted_vote_score  1000 non-null   float64
dtypes: bool(1), float64(1), int64(4), object(2)
memory usage: 55.8+ KB


appid: The unique steam id for each game   
name: The unique game name  
review: The corpus of all text in the review  
timestamp_created: Unix time (epoch) in UTC (POSIX TIME) seconds since 01/01/1970  
`voted_up`: The reviewer's thumb up (True) or thumb down (False) Score `(our target)`  
votes_up: Number of people who upvoted the review  
votes_funny: Number of people who thought the vote was funny  
weighted_vote_score: Steam's helpfullness score - between 0 to 1 - where low scores are likely spam  


In [13]:
df.weighted_vote_score.describe()

count    1000.000000
mean        0.501526
std         0.022687
min         0.254542
25%         0.500000
50%         0.500000
75%         0.500000
max         0.769796
Name: weighted_vote_score, dtype: float64

Looking at the weighted score - we can see the first 75% essentially stay at or below the 50% probability of spam. Why don't we only grab reviews that are substantial by filtering the df to where the score is above 0.50.

In [14]:
df_filtered = df[df.weighted_vote_score > 0.50]
len(df_filtered)

137

Ok, that took us from almost 1k results down to ~130.. we probably should request around 1000 with this criteria before filtering - but let's observe some of the comments in the filtered. 


In [15]:
df_filtered.head()

,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
1,730,Counter-Strike: Global Offensive,"Good game, there are nuances, yes. As in any o...",1753397848,True,1,0,0.523810
20,578080,PUBG: BATTLEGROUNDS,KRAFTON L,1753394676,False,1,0,0.523810
29,578080,PUBG: BATTLEGROUNDS,ITS A GOOD GAME BUT THE HACKING PROBLEM IS INS...,1753337113,True,1,0,0.523810
30,1623730,Palworld,10/10 its like ark but with pokemon slavery!,1753394991,True,1,0,0.509476
32,1623730,Palworld,"What can be really said, the game is gimmicky ...",1753393722,True,1,0,0.509476


Putting these filters in place as a criteria for downloading and saving a revew we get:

In [16]:
# Pre-filter Reviews
quality_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    n_quality = 0
    appid = row['appid']
    name = row['name']
    cursor = '*' # Initial cursor
    
    while n_quality < 10:
        reviews_data = get_reviews(appid, num_reviews=90, cursor=cursor)
        reviews = reviews_data['reviews']
        cursor = reviews_data.get('cursor') # Update's cursor value

        if not reviews:
            break # early escape if no more reviews to fetch
        
        for review in reviews:
            if float(review['weighted_vote_score']) >= 0.51:
                n_quality += 1
                quality_reviews.append({
                    'appid': appid,
                    'name': name,
                    'review': review['review'],
                    'timestamp_created': review['timestamp_created'],
                    'voted_up': review['voted_up'],
                    'votes_up': review['votes_up'],
                    'votes_funny': review['votes_funny'],
                    'weighted_vote_score': review['weighted_vote_score'],
                })
            

quality_reviews_df = pd.DataFrame(all_reviews)
quality_reviews_df.head()

100%|██████████| 100/100 [00:55<00:00,  1.81it/s]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,top game no cheaters,1753398278,True,0,0,0.5
1,730,Counter-Strike: Global Offensive,"Good game, there are nuances, yes. As in any o...",1753397848,True,1,0,0.523809552192687988
2,730,Counter-Strike: Global Offensive,https://www.youtube.com/watch?v=Fqv_qIuyeuU,1753397629,True,0,0,0.5
3,730,Counter-Strike: Global Offensive,Cheat feast,1753397198,False,0,0,0.5
4,730,Counter-Strike: Global Offensive,ytr,1753397066,True,0,0,0.5


# Downloaded QualityReviews

In [17]:
# check in on counts and if filtering now
print("Total reviews:", len(quality_reviews_df))

has_word = quality_reviews_df['review'].str.contains(r'\b\w+\b')
is_long = quality_reviews_df.review.str.len().ge(10)
english_only = quality_reviews_df['review'].str.fullmatch(r"[A-Za-z0-9\s.,!?\"'’\-():;]+", na=False)

filtered_df = quality_reviews_df[has_word & is_long & english_only]
print("Filtered reviews (≥10 chars and has english words):", len(filtered_df))

Total reviews: 1000
Filtered reviews (≥10 chars and has english words): 638


In [18]:
# See how many reviews are left for the 100 games downloaded
min_df = filtered_df
min_df.name.value_counts()

name
Half-Life 2                     10
New World: Aeternum             10
Ring of Elysium                 10
Apex Legends                     9
World of Warships                9
                                ..
Mount & Blade II: Bannerlord     4
Rocket League                    3
Terraria                         3
Wallpaper Engine                 3
Trove                            3
Name: count, Length: 100, dtype: int64

We run into serious memory load issues with the random review length - let's filter out really long reviews 

In [19]:
short_min_df = min_df[min_df.review.str.len() < 50]

print(len(min_df))
print(len(short_min_df))


638
293


In [20]:
# Add a column for review length (optional, for reuse or inspection)
min_df.loc[:, "review_length"] = min_df["review"].str.len()

# Get the longest review in min_df
with pd.option_context("display.max_colwidth", 200):
    print("============ before filter ============")
    display(min_df.sort_values(by="review_length", ascending=False).head(1))

# Same for short_min_df
short_min_df = min_df[min_df["review_length"] < 50]
with pd.option_context("display.max_colwidth", 200):
    print("============ after filter ============")
    display(short_min_df.sort_values(by="review_length", ascending=False).head(1))

============ before filter ============


/tmp/ipykernel_19/2979469817.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  min_df.loc[:, "review_length"] = min_df["review"].str.len()


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score,review_length
757,1665460,eFootball,This is obviously the worst PES game I've ever played. But it shows that Konami wants to get away from the recycle-old-game-and-stick-a-new-name-on scheme. Some ideas and approaches are very good ...,1636486214,True,0,0,0.5,3428


============ after filter ============


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score,review_length
526,582010,Monster Hunter: World,I liked this game it's not boring like other game,1753336175,True,0,1,0.5,49


now we have 952 reviews with a max string length of 49 characters - that's pretty good!
There's some randomization as more reviews are added over time, but this is an excelent starting point

In [21]:
short_min_df.to_csv("quality_reviews.csv")

Ok, now we have a reasonable amount of data with some context around the game, the review, and the voted up tag.  

We're going to generate a corpus of input and then split the full data into 3 sets- train, validate and test - in 80:10:10 batches

# Start Here for Clean Analysis


In [22]:
# Start here for clean
import pandas as pd
min_df = pd.read_csv("quality_reviews.csv")
min_df.head()

,Unnamed: 0,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score,review_length
0,0,730,Counter-Strike: Global Offensive,top game no cheaters,1753398278,True,0,0,0.5,20
1,3,730,Counter-Strike: Global Offensive,Cheat feast,1753397198,False,0,0,0.5,11
2,10,1172470,Apex Legends,this game sucks and i love it,1752688287,True,0,0,0.5,29
3,12,1172470,Apex Legends,Good Shit!,1751979827,True,0,0,0.5,10
4,14,1172470,Apex Legends,Best gunfeel across any FPS I have touched.,1751223704,True,0,0,0.5,43


In [23]:
# Clean the punctuation
import re

def cleaned(text):
    return re.sub(r'\W+', '_', text).lower()

In [24]:
# Add input Field
df = min_df.copy().reset_index(drop=True)
df['input'] = 'TEXT1: ' + df.review

# Convert the target to boolean ints
# Our target is currently saved as boolean true false - so let's convert to int
df['voted_up_int'] = df['voted_up'].astype(int)

display(df['input'].head())
display(df.voted_up.head())

0                          TEXT1: top game no cheaters
1                                   TEXT1: Cheat feast
2                 TEXT1: this game sucks and i love it
3                                    TEXT1: Good Shit!
4    TEXT1: Best gunfeel across any FPS I have touc...
Name: input, dtype: object

0     True
1    False
2     True
3     True
4     True
Name: voted_up, dtype: bool

we might have an oversaturation of voted up = true, let's check

In [25]:
df.voted_up.value_counts()

voted_up
True     259
False     34
Name: count, dtype: int64

Yes - with a very popular game there are only 167 votes down vs 785 votes up. This could influence our fine tuning.

In [26]:
# Now let's get experience with datasets ( required for hugging face transformers )
from datasets import Dataset,DatasetDict

# Select the columns we want to keep for the dataset/prediction purposes
columns = ['input', 'voted_up_int']
df = df[columns].copy()

ds = Dataset.from_pandas(df)

In [27]:
ds

Dataset({
    features: ['input', 'voted_up_int'],
    num_rows: 293
})

In [28]:
# We're going to create todenizers using deberta
model_name = 'microsoft/deberta-v3-small'

!pip install -Uq transformers
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_name)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.1 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


The warning is only an issue because we're using noisy or multilingual data - the reviews, even when marked english, have all kinds of randomness.

Our options are to ignore (if we're ok with slighly less flexible outcomes) - instead of <unk> the characters would be byte level split.  
Or we can use something like..  
> from transformers import T5Tokenizer  
> tokenizer = T5Tokenizer.from_pretrained("t5-base", use_fast=False)  

In [29]:
# Test out the tokenizer with some basic text
tokz.tokenize("TEXT1: An Hello, I'm new at this and learning is fun!")

['▁TEXT',
 '1',
 ':',
 '▁An',
 '▁Hello',
 ',',
 '▁I',
 "'",
 'm',
 '▁new',
 '▁at',
 '▁this',
 '▁and',
 '▁learning',
 '▁is',
 '▁fun',
 '!']

In [30]:
# Vs some other text in the head of an earlier preview
tokz.tokenize("Tässä pelissä on intensiivistä väkivaltaa")

['▁T',
 'ä',
 's',
 's',
 'ä',
 '▁pe',
 'liss',
 'ä',
 '▁on',
 '▁in',
 't',
 'ensi',
 'ivist',
 'ä',
 '▁vä',
 'k',
 'ival',
 'ta',
 'a']

The tokenizer works fine on normal text but not fine on other text

In [31]:
# simple function to tokenize our
def tok_func(x): return tokz(x["input"])

In [32]:
# Parallel for every row in ds with map
tok_ds = ds.map(tok_func, batched=True)

Map:   0%|          | 0/293 [00:00<?, ? examples/s]

In [33]:
row = tok_ds[0]
row['input'], row['input_ids']

('TEXT1: top game no cheaters', [1, 54453, 435, 294, 530, 522, 363, 83303, 2])

These ids are a list of vocab in the tokenizer with a unique int for every string

In [34]:
# See the int above
tokz.vocab['forever']

67551

In [35]:
# Transformers needs a column called labels
tok_ds = tok_ds.rename_columns({'voted_up_int':'labels'})

In [36]:
tok_ds

Dataset({
    features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 293
})

`tok_ds` is now ready for splitting

In [37]:
# Step 1: Train/Test Split (e.g., 80% train, 20% temp)
train_test = tok_ds.train_test_split(test_size=0.2, seed=42)

# Step 2: Split test portion into validation and test (e.g., 50/50 of the 20%)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_ds = train_test['train']
test_ds = val_test['train']
eval_ds = val_test['test']

dds = DatasetDict({
    'train': train_ds,
    'test': test_ds,
    'eval': eval_ds
})

In [38]:
dds

DatasetDict({
    train: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 234
    })
    test: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 29
    })
    eval: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30
    })
})

In [39]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model_name = 'microsoft/deberta-v3-small'
tokz = AutoTokenizer.from_pretrained(model_name)

# Define metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Load model and enable checkpointing
from transformers import DebertaV2ForSequenceClassification

model = DebertaV2ForSequenceClassification.from_pretrained(model_name, num_labels=2)


# Define training arguments
args = TrainingArguments(
    output_dir='outputs',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    gradient_checkpointing=False,
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,   # Make sure you define these
    eval_dataset=eval_ds,
    tokenizer=tokz,
    compute_metrics=compute_metrics
)

trainer.train()


2025-07-25 00:09:50.841773: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753402191.034680      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753402191.090034      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_19/1352258326.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.243823,0.933333,0.965517
2,No log,0.241293,0.933333,0.965517
3,No log,0.234064,0.933333,0.965517


TrainOutput(global_step=90, training_loss=0.4237044440375434, metrics={'train_runtime': 11.7587, 'train_samples_per_second': 59.701, 'train_steps_per_second': 7.654, 'total_flos': 2786571685080.0, 'train_loss': 0.4237044440375434, 'epoch': 3.0})

In [40]:
# Get raw logits
raw_preds = trainer.predict(eval_ds)

# Convert logits to class predictions (0 or 1)
preds = np.argmax(raw_preds.predictions, axis=1)

# Get ground truth labels
true_labels = raw_preds.label_ids

# Optionally inspect predictions and labels
print("Predictions:", preds[:10])
print("True Labels:", true_labels[:10])

# Compute accuracy
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(true_labels, preds)
print(f"\nAccuracy: {accuracy:.4f}")

# Optional: print precision, recall, F1
print("\nClassification Report:")
print(classification_report(true_labels, preds))

Predictions: [1 1 1 1 1 1 1 1 1 1]
True Labels: [1 1 1 1 1 1 0 1 1 1]

Accuracy: 0.9333

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.93      1.00      0.97        28

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


As we expected earlier on - because the dataset was oversaturated with upvotes, the fine tuning did much better on the upvotes - the 1's in the report above with P/R both above 90%.  

Where the 0's in the report above fell to precision ~70 and recall ~60%

Meaning we correctly predicted downvotes 69% of the time, catching 60% of the downvotes, and  
Correctly predicted upvotes 93% of the time, catching 95% of the upvotes.

Overall this gives us an accuracy rating of 90%

In [41]:
# Get predictions from trainer
raw_preds = trainer.predict(eval_ds)
preds = np.argmax(raw_preds.predictions, axis=1)
true_labels = raw_preds.label_ids

# Convert dataset to pandas (only if it's not already)
df_eval = eval_ds.to_pandas()

# Add predictions and true labels to DataFrame
df_eval["predicted"] = preds
df_eval["label"] = true_labels

In [42]:
# Preview
with pd.option_context("display.max_colwidth", 200):
    display(df_eval)

,input,labels,input_ids,token_type_ids,attention_mask,predicted,label
0,TEXT1: Good Shit!,1,"[1, 54453, 435, 294, 1798, 55819, 300, 2]","[0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1]",1,1
1,TEXT1: i love the game i can't wait for the second one\r\n,1,"[1, 54453, 435, 294, 584, 472, 262, 522, 584, 295, 280, 297, 1495, 270, 262, 567, 311, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
2,TEXT1: great world war 2 game,1,"[1, 54453, 435, 294, 426, 447, 1442, 392, 522, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
3,TEXT1: Best game ever,1,"[1, 54453, 435, 294, 1652, 522, 632, 2]","[0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1]",1,1
4,TEXT1: Best game ever.,1,"[1, 54453, 435, 294, 1652, 522, 632, 260, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
5,TEXT1: yes portugal,1,"[1, 54453, 435, 294, 2489, 115321, 2]","[0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1]",1,1
6,TEXT1: I am happy I graduate from Destiny,0,"[1, 54453, 435, 294, 273, 481, 1005, 273, 3665, 292, 21778, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,0
7,TEXT1: i love arthur,1,"[1, 54453, 435, 294, 584, 472, 973, 29999, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1
8,TEXT1: great game,1,"[1, 54453, 435, 294, 426, 522, 2]","[0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1]",1,1
9,TEXT1: Just the best game of all the time imo.,1,"[1, 54453, 435, 294, 1063, 262, 410, 522, 265, 305, 262, 326, 45564, 260, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1


Overall not too bad for a first pass with chaotic data.